In [2]:
import sqlite3
import plotly.express
import pandas
import plotly.graph_objects
import numpy

# column_to_display = 'num_evaluation_timesteps'
column_to_display = 'num_function_evaluations'

# Connect to the SQLite database
db_path = '/home/dimitri/code/oll_onemax/computed/fire/dotted.db'
conn = sqlite3.connect(db_path)

# Load data from the EVALUATION_EPISODES table with variance calculation
query_evaluation = f"""
SELECT policy_id,
       AVG({column_to_display}) AS avg_value,
       COUNT(*) AS row_count,
       AVG(({column_to_display} - avg_value) * ({column_to_display} - avg_value)) AS var_value
FROM (
    SELECT policy_id,
           {column_to_display},
           AVG({column_to_display}) OVER (PARTITION BY policy_id) AS avg_value
    FROM EVALUATION_EPISODES
)
GROUP BY policy_id
"""
df_evaluation = pandas.read_sql_query(query_evaluation, conn)
df_evaluation.set_index('policy_id', inplace=True)

# Load num_total_timesteps from the CONSTRUCTED_POLICIES table
query_policies = """
SELECT policy_id, num_total_timesteps
FROM CONSTRUCTED_POLICIES
"""
df_policies = pandas.read_sql_query(query_policies, conn)

# Merge the two dataframes on policy_id
df_merged = df_evaluation.merge(df_policies.drop_duplicates(subset='policy_id'), on='policy_id', how='left')
df_merged.set_index('policy_id', inplace=True)

# Close the connection
conn.close()

# Calculate standard deviation
df_merged['stddev_value'] = numpy.sqrt(df_merged['var_value'])

# Separate the baseline data (policy_id = -1)
df_baseline = df_merged[df_merged.index == -1]
df_others = df_merged[df_merged.index != -1]

# Create the line plot for other policies
fig = plotly.express.line(
    df_others,
    x='num_total_timesteps',
    y='avg_value',
    labels={'num_total_timesteps': 'Number of Total Timesteps', 'avg_value': f'Average {column_to_display.replace("_", " ").title()}'},
    title=f'Average {column_to_display.replace("_", " ").title()} by Number of Total Timesteps'
)

# Update the plot with hover data for other policies
fig.update_traces(
    mode='markers+lines',
    hovertemplate='<b>Number of Total Timesteps:</b> %{x}<br>' +
                  '<b>Average Value:</b> %{y}<br>' +
                  '<b>Row Count:</b> %{customdata[0]}<br>' +
                  '<b>Standard Deviation:</b> %{customdata[1]}'
)

# Add hover data for other policies
fig.update_traces(customdata=df_others[['row_count', 'stddev_value']])

# Add the standard deviation shading for other policies
fig.add_traces([
    plotly.graph_objects.Scatter(
        x=df_others['num_total_timesteps'],
        y=df_others['avg_value'] + df_others['stddev_value'],
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip'
    ),
    plotly.graph_objects.Scatter(
        x=df_others['num_total_timesteps'],
        y=df_others['avg_value'] - df_others['stddev_value'],
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(0,100,80,0.2)',
        showlegend=False,
        hoverinfo='skip'
    )
])

# Add the baseline line with hoverable points and standard deviation shading
if not df_baseline.empty:
    baseline_value = df_baseline['avg_value'].values[0]
    baseline_stddev = numpy.sqrt(df_baseline['var_value'].values[0])

    fig.add_trace(
        plotly.graph_objects.Scatter(
            x=df_others['num_total_timesteps'],
            y=[baseline_value] * len(df_others),
            mode='lines',
            line=dict(color='orange', dash='dash'),
            name='Baseline',
            hoverinfo='y',
            hovertemplate='<b>Baseline:</b><br>' +
                          '<b>Average Value:</b> %{y}<br>' +
                          f'<b>Standard Deviation:</b> {baseline_stddev}'
        )
    )

    fig.add_traces([
        plotly.graph_objects.Scatter(
            x=df_others['num_total_timesteps'],
            y=[baseline_value + baseline_stddev] * len(df_others),
            mode='lines',
            line=dict(width=0),
            showlegend=False,
            hoverinfo='skip'
        ),
        plotly.graph_objects.Scatter(
            x=df_others['num_total_timesteps'],
            y=[baseline_value - baseline_stddev] * len(df_others),
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor='rgba(255,165,0,0.2)',
            showlegend=False,
            hoverinfo='skip'
        )
    ])

# Update layout to start y-axis from 0
fig.update_layout(yaxis=dict(range=[0, None]))

# Show the plot
fig.show()